# Knee MRI submission: ConvNeXt-T 20 epochs (`v6cnxT20`), with timing
Inputs to attach: competition data, `knee-mri-pipeline` dataset (patched **v6_3** code: `infer.py` must know `head`), and the notebook output that holds `models_v6cnxT20/fold{0..4}_v6cnxT20.pt`.
Later, to ensemble the second seed, attach its output too and set `MODEL_TAGS = ["v6cnxT20", "v6cnxT20s43"]` in cell 1.
GPU on, internet OFF. Nothing is downloaded: every model is built from the weights stored in its checkpoint.
Every number printed under TIMING is measured. On the visible commit run the test set is tiny, so cell 4 benchmarks 100 TRAIN studies and projects the full-scoring time; that benchmark is skipped automatically when the test set has 50+ studies (the real scoring run).
Same inference code and flags as the earlier submissions (`infer.py --workers 4 --bs 4`, rank average), so the public score is comparable with sub6 / sub9.

In [1]:
import os, re, sys, time, json, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
T0 = time.time()
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working")); W.mkdir(parents=True, exist_ok=True)

MODEL_TAGS = ["v6cnxT20"]            # add "v6cnxT20s43" once that run is attached; every tag needs folds 0..4
CODE_OVERRIDE = None                 # folder containing infer.py, model.py, cache_data.py (set if "ambiguous")
MODEL_DIR_OVERRIDE = {}              # e.g. {"v6cnxT20": "/kaggle/input/notebooks/.../models_v6cnxT20"}
LABELS_OVERRIDE = None               # OLD labels_soft.csv, only for fallback priors of unreadable studies (default: not used, those get 0.5)
USE_PRIORS = False                   # the handoff spec says do not pass --labels; set True only if you want the training prior instead of 0.5
N_TEST_FULL_EST = 1300               # assumed hidden-test size, only for the projection

SKIP = {"train_series", "test_series", "cache", "__pycache__", ".git"}
def walk(root, maxdepth=7):
    q = [(Path(root), 0)]
    while q:
        d, k = q.pop(0)
        try: ents = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError: continue
        for e in ents:
            if e.is_dir(follow_symlinks=False):
                if e.name not in SKIP and k < maxdepth: q.append((Path(e.path), k + 1))
            else: yield Path(e.path)
files = list(walk(ROOT)); print("input files scanned:", len(files))

def pick(cands, what, override):
    if override: return Path(override)
    cands = sorted(set(cands)); print(f"{what}: {len(cands)} found", *[f"\n   {c}" for c in cands])
    assert len(cands) == 1, f"ambiguous or missing {what}: set the override variable above"
    return cands[0]

code_dirs = [f.parent for f in files if f.name == "infer.py" and (f.parent / "model.py").exists() and (f.parent / "cache_data.py").exists()]
CODE = pick(code_dirs, "code folder (infer.py)", CODE_OVERRIDE)
assert "KneeNet" in (CODE / "model.py").read_text(), "model.py has no KneeNet"
assert '"head"' in (CODE / "infer.py").read_text() or "'head'" in (CODE / "infer.py").read_text(), "infer.py does not know the tokens head: attach the v6_3 dataset version"
assert (CODE / "geom.py").exists(), "geom.py missing: not the v6 code"
data_dirs = [f.parent for f in files if f.name == "test_series.csv" and (f.parent / "sample_submission.csv").exists() and (f.parent / "test.csv").exists()]
DATA = pick(data_dirs, "competition data folder", None)

CK = {}                                  # tag -> {fold: path}
for tag in MODEL_TAGS:
    if tag in MODEL_DIR_OVERRIDE:
        found = {int(re.fullmatch(rf"fold(\d)_{tag}\.pt", p.name).group(1)): p for p in Path(MODEL_DIR_OVERRIDE[tag]).glob(f"fold?_{tag}.pt")}
    else:
        by_dir = {}
        for f in files:
            m = re.fullmatch(rf"fold(\d)_{re.escape(tag)}\.pt", f.name)
            if m: by_dir.setdefault(f.parent, {})[int(m.group(1))] = f
        print(f"{tag}: folders", {str(k): sorted(v) for k, v in by_dir.items()})
        assert len(by_dir) == 1, f"{tag}: found in {len(by_dir)} folders: set MODEL_DIR_OVERRIDE['{tag}']"
        found = next(iter(by_dir.values()))
    assert sorted(found) == [0, 1, 2, 3, 4], f"{tag}: folds found {sorted(found)} (need 0..4)"
    CK[tag] = found
LABELS = None
if USE_PRIORS:
    cand = [f for f in files if f.name == "labels_soft.csv"]
    LABELS = Path(LABELS_OVERRIDE) if LABELS_OVERRIDE else (cand[0] if len(cand) == 1 else None)
print("fallback-prior labels:", LABELS, "(None -> unreadable studies get 0.5)")
print(f"\nSETUP OK  [{time.time()-T0:.0f}s]")

input files scanned: 49
code folder (infer.py): 1 found 
   /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline
competition data folder: 1 found 
   /kaggle/input/competitions/rsna-knee-abnormality-detection
v6cnxT20: folders {'/kaggle/input/notebooks/theokorir/knee-v6-cnxt20-5fold/models_v6cnxT20': [0, 1, 2, 3, 4]}
fallback-prior labels: None (None -> unreadable studies get 0.5)

SETUP OK  [0s]


In [2]:
# ---- assemble ONE folder of checkpoints with unique names and verify them ----
import torch
ALL = W / "models_all"; shutil.rmtree(ALL, ignore_errors=True); ALL.mkdir()
rows = []
for tag, fm in CK.items():
    for i in range(5):
        dst = ALL / f"fold{i}_{tag}.pt"; shutil.copy(fm[i], dst)
        ck = torch.load(dst, map_location="cpu")
        rows.append(dict(file=dst.name, backbone=ck["backbone"], K=ck["K"], head=ck.get("head", "series"), canon=ck.get("canon", False),
                         n_tensors=len(ck["state"]), MB=round(dst.stat().st_size / 1e6, 1)))
M = pd.DataFrame(rows); print(M.to_string(index=False))
assert len(M) == 5 * len(CK), "unexpected number of checkpoints"
assert M.K.nunique() == 1, "checkpoints disagree on K: infer.py decodes with the LAST checkpoint's K only"
assert (M["head"] == "tokens").all(), "expected token-head models only"
print(f"\n{len(M)} checkpoints OK | backbone(s) {sorted(M.backbone.unique())} | K {M.K[0]} | canon {sorted(set(M.canon))}")

             file                    backbone  K   head  canon  n_tensors    MB
fold0_v6cnxT20.pt convnext_tiny.in12k_ft_in1k  1 tokens  False        213 115.5
fold1_v6cnxT20.pt convnext_tiny.in12k_ft_in1k  1 tokens  False        213 115.5
fold2_v6cnxT20.pt convnext_tiny.in12k_ft_in1k  1 tokens  False        213 115.5
fold3_v6cnxT20.pt convnext_tiny.in12k_ft_in1k  1 tokens  False        213 115.5
fold4_v6cnxT20.pt convnext_tiny.in12k_ft_in1k  1 tokens  False        213 115.5

5 checkpoints OK | backbone(s) ['convnext_tiny.in12k_ft_in1k'] | K 1 | canon [False]


In [3]:
# ---- run inference (timed) ----
test_ids = pd.read_csv(DATA / "test.csv").StudyInstanceUID
N_TEST = len(test_ids); print("test studies in this run:", N_TEST)

def run_infer(data_dir, out, tag):
    cmd = [sys.executable, str(CODE / "infer.py"), "--models_dir", str(ALL), "--data_dir", str(data_dir),
           "--out", str(out), "--workers", "4", "--bs", "4"]
    if LABELS: cmd += ["--labels", str(LABELS)]
    t = time.time(); last = ""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(CODE))
    for line in p.stdout:
        last = line.rstrip()
        if "/" in last and "|" in last and last.split("/")[0].isdigit():
            if int(last.split("/")[0]) % 40 == 0: print(f"[{tag}] {last}", flush=True)
        else: print(f"[{tag}] {last}", flush=True)
    rc = p.wait(); wall = time.time() - t
    assert rc == 0, f"infer.py failed (exit {rc})"
    m = re.search(r"\| (\d+)s\s*$", last); loop = float(m.group(1)) if m else float("nan")
    return wall, loop

wall_main, loop_main = run_infer(DATA, W / "submission.csv", "main")
print(f"\nmain run: wall {wall_main:.0f}s | inference loop {loop_main:.0f}s | load/setup ~{wall_main-loop_main:.0f}s | per study {loop_main/max(N_TEST,1):.2f}s")

test studies in this run: 3
[main] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[main]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[main] models: 5 | canonical-geometry models: 0 | token-head models: 5
[main] saved /kaggle/working/submission.csv (3, 13) | unreadable studies: 0 | 13s

main run: wall 35s | inference loop 13s | load/setup ~22s | per study 4.33s


In [4]:
# ---- benchmark for the projection: ONLY when the test set is tiny (commit run). Skipped in the real scoring run. ----
BENCH_N = 100
if N_TEST < 50:
    tr = pd.read_csv(DATA / "train.csv"); ts = pd.read_csv(DATA / "train_series.csv")
    ids = tr.StudyInstanceUID.sample(BENCH_N, random_state=0).tolist()
    B = Path("/tmp/bench"); shutil.rmtree(B, ignore_errors=True); B.mkdir(parents=True)
    pd.DataFrame({"StudyInstanceUID": ids}).to_csv(B / "test.csv", index=False)
    ts[ts.StudyInstanceUID.isin(ids)].to_csv(B / "test_series.csv", index=False)
    sub0 = pd.read_csv(DATA / "sample_submission.csv"); cols = list(sub0.columns)
    sb = pd.DataFrame(0.5, index=range(BENCH_N), columns=cols); sb["StudyInstanceUID"] = ids; sb.to_csv(B / "sample_submission.csv", index=False)
    os.symlink(DATA / "train_series", B / "test_series")
    wall_b, loop_b = run_infer(B, B / "bench_sub.csv", "bench")
    per = loop_b / BENCH_N; load_s = wall_b - loop_b; proj = load_s + per * N_TEST_FULL_EST
    print(f"\n===== TIMING PROJECTION ({len(M)} models: {', '.join(CK)}, THIS GPU/CPU) =====")
    print(f"  model load + setup : {load_s:6.0f} s")
    print(f"  per study          : {per:6.2f} s   ({BENCH_N} train studies)")
    print(f"  projected for {N_TEST_FULL_EST} studies: {proj:7.0f} s = {proj/3600:.2f} h   (limit 9 h = 32400 s)")
    print(f"  efficiency runtime term RuntimeSeconds/32400 = {proj/32400:.3f}")
    print(f"  rule of thumb: +0.01 AUC is worth ~900-1000 s of runtime on the Efficiency Track")
    print("  caveat: scoring hardware may differ from this run; train studies may decode a bit differently from test.")
else:
    print(f"test set has {N_TEST} studies: real scoring run, benchmark skipped.")

[bench] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[bench]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[bench] models: 5 | canonical-geometry models: 0 | token-head models: 5
[bench] 40/100 | 40s
[bench] 80/100 | 73s
[bench] saved /tmp/bench/bench_sub.csv (100, 13) | unreadable studies: 0 | 91s

===== TIMING PROJECTION (5 models: v6cnxT20, THIS GPU/CPU) =====
  model load + setup :     15 s
  per study          :   0.91 s   (100 train studies)
  projected for 1300 studies:    1198 s = 0.33 h   (limit 9 h = 32400 s)
  efficiency runtime term RuntimeSeconds/32400 = 0.037
  rule of thumb: +0.01 AUC is worth ~900-1000 s of runtime on the Efficiency Track
  caveat: scoring hardware may differ from this run; train studies may decode a bit differently from test.


In [5]:
# ---- final validation of submission.csv ----
sub = pd.read_csv(W / "submission.csv"); ref = pd.read_csv(DATA / "sample_submission.csv")
assert list(sub.columns) == list(ref.columns), "columns differ from sample_submission"
assert set(sub.StudyInstanceUID) == set(ref.StudyInstanceUID) and len(sub) == len(ref), "study ids differ"
vals = sub.drop(columns="StudyInstanceUID")
assert vals.notna().all().all() and ((vals >= 0) & (vals <= 1)).all().all(), "NaN or out-of-range values"
print(f"({len(sub)}, {sub.shape[1]}) OK | min std of predictions per label (should not be ~0):", vals.std().round(3).min().round(3))
print(f"TOTAL notebook time so far: {time.time()-T0:.0f}s")

AttributeError: 'float' object has no attribute 'round'